In [1]:
"""
coalesce() vs repartition() — PySpark 4.0
==========================================
Each section isolates one behavioural difference and uses explain()
or partition-size inspection to make it observable.

Key mental model
  coalesce(n)     — narrow transformation; merges adjacent partitions;
                    never shuffles; can only REDUCE partition count.
  repartition(n)  — wide transformation; full shuffle via RoundRobin or
                    hash; can INCREASE or DECREASE partition count;
                    always produces even distribution.
"""

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
def section(title: str) -> None:
    bar = "=" * 70
    print(f"\n{bar}\n  {title}\n{bar}\n")


def partition_sizes(df) -> list[int]:
    """Row count per partition."""
    return df.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()


# local[8] = 8 threads in one JVM (driver + executor in the same process).
# Plans, partition counts, and stage boundaries are identical to a cluster;
# shuffle cost is local-disk I/O only — no network overhead.
spark = (
    SparkSession.builder
    .appName("CoalesceVsRepartition")
    .master("local[8]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/02 00:35:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 1. coalesce() IS A NARROW TRANSFORMATION — NO SHUFFLE
#
#  coalesce() merges existing partitions locally on each executor.
#  No data crosses the network, so there is no Exchange node in the plan.
# ═══════════════════════════════════════════════════════════════════════════ #
section("1. coalesce() — narrow transformation, no Exchange")
print("""
  Start with 8 partitions, coalesce to 3.

  Look for:  NO Exchange node in the physical plan.
             Partition count drops without a shuffle stage.
""")

df8 = spark.range(10_000).repartition(8)
print(f"before coalesce: {df8.rdd.getNumPartitions()} partitions")

coalesced = df8.coalesce(3)
print(f"after  coalesce: {coalesced.rdd.getNumPartitions()} partitions\n")

coalesced.explain("formatted")


  1. coalesce() — narrow transformation, no Exchange


  Start with 8 partitions, coalesce to 3.

  Look for:  NO Exchange node in the physical plan.
             Partition count drops without a shuffle stage.

before coalesce: 8 partitions
after  coalesce: 3 partitions

== Physical Plan ==
AdaptiveSparkPlan (8)
+- == Final Plan ==
   ResultQueryStage (5)
   +- Coalesce (4)
      +- ShuffleQueryStage (3), Statistics(sizeInBytes=156.3 KiB, rowCount=1.00E+4)
         +- Exchange (2)
            +- * Range (1)
+- == Initial Plan ==
   Coalesce (7)
   +- Exchange (6)
      +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 10000, step=1, splits=Some(8))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=32]

(3) ShuffleQueryStage
Output [1]: [id#0L]
Arguments: 0

(4) Coalesce
Input [1]: [id#0L]
Arguments: 3

(5) ResultQueryStage
Output [1]: [id#0L]
Arguments: 1

(6) Exchange
Input [1]: [id#0L]
Arguments: Roun

In [4]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 2. repartition() IS A WIDE TRANSFORMATION — ALWAYS SHUFFLES
#
#  repartition() inserts a full shuffle regardless of whether the count
#  goes up or down.  Every row is rehashed and sent across the network.
# ═══════════════════════════════════════════════════════════════════════════ #
section("2. repartition() — wide transformation, Exchange always present")
print("""
  Same 8-partition DataFrame, repartition to 3.

  Look for:  Exchange RoundRobinPartitioning(3) in the physical plan.
             This is the shuffle stage — data moves across executors.
""")

repartitioned = df8.repartition(3)
print(f"after repartition: {repartitioned.rdd.getNumPartitions()} partitions\n")

repartitioned.explain("formatted")


  2. repartition() — wide transformation, Exchange always present


  Same 8-partition DataFrame, repartition to 3.

  Look for:  Exchange RoundRobinPartitioning(3) in the physical plan.
             This is the shuffle stage — data moves across executors.

after repartition: 3 partitions

== Physical Plan ==
AdaptiveSparkPlan (6)
+- == Final Plan ==
   ResultQueryStage (4)
   +- ShuffleQueryStage (3), Statistics(sizeInBytes=156.3 KiB, rowCount=1.00E+4)
      +- Exchange (2)
         +- * Range (1)
+- == Initial Plan ==
   Exchange (5)
   +- Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 10000, step=1, splits=Some(8))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=53]

(3) ShuffleQueryStage
Output [1]: [id#0L]
Arguments: 0

(4) ResultQueryStage
Output [1]: [id#0L]
Arguments: 1

(5) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=48]

(6) AdaptiveSparkPla

In [5]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 3. coalesce() CANNOT INCREASE PARTITION COUNT
#
#  Requesting more partitions than currently exist is silently clamped
#  to the current count.  repartition() has no such restriction.
# ═══════════════════════════════════════════════════════════════════════════ #
section("3. coalesce() silently ignores requests to increase partitions")
print("""
  Start with 4 partitions.  Ask coalesce(100) — still 4.
  Ask repartition(100)      — becomes 100.
""")

df4 = spark.range(1_000).repartition(4)
print(f"source partitions          : {df4.rdd.getNumPartitions()}")
print(f"after coalesce(100)        : {df4.coalesce(100).rdd.getNumPartitions()}  ← clamped")
print(f"after repartition(100)     : {df4.repartition(100).rdd.getNumPartitions()}  ← increased")


  3. coalesce() silently ignores requests to increase partitions


  Start with 4 partitions.  Ask coalesce(100) — still 4.
  Ask repartition(100)      — becomes 100.

source partitions          : 4
after coalesce(100)        : 4  ← clamped
after repartition(100)     : 100  ← increased


In [8]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 4. DATA DISTRIBUTION — repartition() is uniform, coalesce() can skew
#
#  coalesce() merges adjacent partitions.  If the source is already skewed,
#  or if the merge pattern is unlucky, output partitions can vary greatly
#  in size.  repartition()'s round-robin shuffle distributes rows evenly.
# ═══════════════════════════════════════════════════════════════════════════ #
section("4. Data distribution — coalesce() can skew, repartition() stays even")
print("""
  Create a DataFrame with unequal partition sizes,
  then collapse to 2 partitions both ways.

  Look for:  coalesce partition sizes that reflect merged adjacent chunks.
             repartition partition sizes that are roughly equal.
""")

# Build skewed source then create unequal sizes by unioning DataFrames of different lengths.
big_chunk   = spark.range(9_000)   # will become partition 0
small_chunk = spark.range(1_000)   # will become partition 1

# Force into exactly 2 partitions by repartitioning each to 1 then unioning
skewed = big_chunk.repartition(1).union(small_chunk.repartition(1))
print(f"source partition sizes : {partition_sizes(skewed)}")

# Both collapse to 1 partition for simplicity of comparison
via_coalesce     = skewed.coalesce(1)
print(f"coalesced partition sizes : {partition_sizes(via_coalesce)}")
via_repartition  = skewed.repartition(1)
print(f"repartitioned partition sizes : {partition_sizes(via_repartition)}")

# Now start from many even partitions and collapse to 3 to show merge pattern
even8 = spark.range(80_000).repartition(8)  # ~10 000 rows each
print(f"\neven source (8 parts)   : {partition_sizes(even8)}")
print(f"coalesce(3) sizes       : {partition_sizes(even8.coalesce(3))}")
print(f"repartition(3) sizes    : {partition_sizes(even8.repartition(3))}")


  4. Data distribution — coalesce() can skew, repartition() stays even


  Create a DataFrame with unequal partition sizes,
  then collapse to 2 partitions both ways.

  Look for:  coalesce partition sizes that reflect merged adjacent chunks.
             repartition partition sizes that are roughly equal.

source partition sizes : [9000, 1000]
coalesced partition sizes : [10000]
repartitioned partition sizes : [10000]

even source (8 parts)   : [10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000]
coalesce(3) sizes       : [20000, 30000, 30000]
repartition(3) sizes    : [26667, 26665, 26668]


In [9]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 5. repartition(n, col) — HASH PARTITIONING BY COLUMN
#
#  repartition() accepts one or more column names.  All rows sharing the
#  same key value are guaranteed to land in the same partition — useful
#  for co-locating data before a join or window function.
#  coalesce() has no column-aware variant.
# ═══════════════════════════════════════════════════════════════════════════ #
section("5. repartition(n, col) — hash partitioning by key")
print("""
  repartition(4, 'dept') routes all rows for a given dept to one partition.

  Look for:  Exchange hashpartitioning(dept, 4) in the physical plan.
             Each partition contains exactly one distinct dept value.
""")

events = (
    spark.range(10_000)
    .withColumn("dept", (F.col("id") % 4).cast("int"))
)

by_dept = events.repartition(4, "dept")
by_dept.explain("formatted")

# Verify: each partition should contain exactly one dept value
distinct_depts_per_partition = (
    by_dept.rdd
    .mapPartitionsWithIndex(
        lambda idx, rows: [(idx, len({r["dept"] for r in rows}))]
    )
    .collect()
)
print("partition → distinct dept values:")
for part, n_depts in distinct_depts_per_partition:
    print(f"  partition {part}: {n_depts} distinct dept(s)")


  5. repartition(n, col) — hash partitioning by key


  repartition(4, 'dept') routes all rows for a given dept to one partition.

  Look for:  Exchange hashpartitioning(dept, 4) in the physical plan.
             Each partition contains exactly one distinct dept value.

== Physical Plan ==
AdaptiveSparkPlan (4)
+- Exchange (3)
   +- Project (2)
      +- Range (1)


(1) Range
Output [1]: [id#101L]
Arguments: Range (0, 10000, step=1, splits=Some(8))

(2) Project
Output [2]: [id#101L, cast((id#101L % 4) as int) AS dept#102]
Input [1]: [id#101L]

(3) Exchange
Input [2]: [id#101L, dept#102]
Arguments: hashpartitioning(dept#102, 4), REPARTITION_BY_NUM, [plan_id=556]

(4) AdaptiveSparkPlan
Output [2]: [id#101L, dept#102]
Arguments: isFinalPlan=false


partition → distinct dept values:
  partition 0: 0 distinct dept(s)
  partition 1: 0 distinct dept(s)
  partition 2: 1 distinct dept(s)
  partition 3: 3 distinct dept(s)


In [10]:
# ═══════════════════════════════════════════════════════════════════════════ #
# 6. THE COALESCE TRAP — coalescing BEFORE a wide op kills parallelism
#
#  Because coalesce() is a narrow transformation, Spark fuses it into the
#  same stage as whatever comes before it.  If you coalesce to N then run
#  a shuffle (groupBy, join…), the shuffle reads from only N partitions.
#
#  Coalescing AFTER a shuffle is always safe — the shuffle already produced
#  spark.sql.shuffle.partitions output partitions; coalescing reduces those
#  without adding another shuffle.
# ═══════════════════════════════════════════════════════════════════════════ #
section("6. The coalesce trap — placement relative to a shuffle matters")
print("""
  We aggregate a 100-partition DataFrame in three ways.
  Watch the partition count feeding into the Exchange (shuffle read side).

  BEFORE (trap)  — coalesce(2) then groupBy  → only 2 tasks in the map stage
  AFTER  (safe)  — groupBy then coalesce(2)  → 8 tasks in the map stage
  repartition    — repartition(2) then groupBy → adds an extra shuffle but
                   preserves parallelism if you need a specific count
""")

source = spark.range(1_000_000).withColumn("key", F.col("id") % 10)
source = source.repartition(100)

print("--- coalesce BEFORE groupBy (trap: map stage runs with 2 tasks) ---")
source.coalesce(2).groupBy("key").count().explain("formatted")

print("--- coalesce AFTER groupBy (safe: map stage runs with 100 tasks) ---")
source.groupBy("key").count().coalesce(2).explain("formatted")

print("--- repartition BEFORE groupBy (extra shuffle but full parallelism) ---")
source.repartition(2).groupBy("key").count().explain("formatted")


  6. The coalesce trap — placement relative to a shuffle matters


  We aggregate a 100-partition DataFrame in three ways.
  Watch the partition count feeding into the Exchange (shuffle read side).

  BEFORE (trap)  — coalesce(2) then groupBy  → only 2 tasks in the map stage
  AFTER  (safe)  — groupBy then coalesce(2)  → 8 tasks in the map stage
  repartition    — repartition(2) then groupBy → adds an extra shuffle but
                   preserves parallelism if you need a specific count

--- coalesce BEFORE groupBy (trap: map stage runs with 2 tasks) ---
== Physical Plan ==
AdaptiveSparkPlan (8)
+- HashAggregate (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Coalesce (4)
            +- Exchange (3)
               +- Project (2)
                  +- Range (1)


(1) Range
Output [1]: [id#103L]
Arguments: Range (0, 1000000, step=1, splits=Some(8))

(2) Project
Output [1]: [(id#103L % 10) AS key#104L]
Input [1]: [id#103L]

(3) Exchange
Input [1]: [key#104L]
Arguments: Roun

In [ ]:
spark.stop()